In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install groq neo4j joblib nltk

import re
import joblib
import nltk
from nltk.corpus import stopwords
from neo4j import GraphDatabase
from groq import Groq
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 27.2 MB/s eta 0:00:00


In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
model_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/intent_model.pkl"
vectorizer_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/vectorizer.pkl"

model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [ ]:
def normalize_intent(intent):
    intent = intent.lower()

    if "lost" in intent or "stolen" in intent:
        return "LostCard"

    if "activate" in intent:
        return "ActivateCard"

    if "close" in intent:
        return "CloseAccount"

    return None

In [ ]:
uri = "neo4j+s://YOUR_DB_ID.databases.neo4j.io"
username = "neo4j"
password = "YOUR_PASSWORD"

In [ ]:
def get_graph_response(intent_name):

    query = """
    MATCH (i:Intent {name:$intent})-[:TRIGGERS]->(s)-[:REQUIRES]->(v)
    RETURN s.name AS service, v.name AS verification
    """

    with driver.session() as session:
        result = session.run(query, intent=intent_name)
        return [record.data() for record in result]

In [ ]:
def build_context(result):

    if not result:
        return "No information available."

    service = result[0]['service']
    verification = result[0]['verification']

    return f"""
    Service: {service}
    Verification: {verification}
    """

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

In [ ]:
def intent_agent(user_query):

    cleaned = clean_text(user_query)
    vec = vectorizer.transform([cleaned])
    predicted_intent = model.predict(vec)[0]

    graph_intent = normalize_intent(predicted_intent)

    return {
        "predicted_intent": predicted_intent,
        "graph_intent": graph_intent
    }

In [ ]:
def graph_agent(intent_data):

    graph_intent = intent_data["graph_intent"]

    if not graph_intent:
        return {"error": "Intent not mapped"}

    graph_result = get_graph_response(graph_intent)

    return {
        "graph_intent": graph_intent,
        "graph_result": graph_result
    }

In [ ]:
def response_agent(user_query, graph_data):

    if "error" in graph_data:
        return "I'm sorry, I couldn't understand your request."

    context = build_context(graph_data["graph_result"])

    prompt = f"""
    You are a banking assistant.

    User query:
    {user_query}

    Knowledge:
    {context}

    Provide a helpful response.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
def agent_pipeline(user_query):

    print("\n--- AGENT PIPELINE START ---")

    # Step 1 — Intent Agent
    intent_data = intent_agent(user_query)
    print("Intent Agent:", intent_data)

    # Step 2 — Graph Agent
    graph_data = graph_agent(intent_data)
    print("Graph Agent:", graph_data)

    # Step 3 — Response Agent
    response = response_agent(user_query, graph_data)

    print("--- AGENT PIPELINE END ---\n")

    return response

In [ ]:
agent_pipeline("My credit card was stolen")


--- AGENT PIPELINE START ---
Intent Agent: {'predicted_intent': 'lost_or_stolen_card', 'graph_intent': 'LostCard'}
Graph Agent: {'graph_intent': 'LostCard', 'graph_result': [{'service': 'CardReplacement', 'verification': 'IdentityVerification'}]}
--- AGENT PIPELINE END ---



"I'm so sorry to hear that your credit card was stolen. Don't worry, I'm here to help. To ensure your account's security, I'll need to verify your identity through our IdentityVerification process. This will help me confirm that I'm speaking with the actual cardholder.\n\nOnce your identity is verified, I can assist you with our CardReplacement service. This will allow us to cancel your stolen card and issue a new one with a new card number, expiration date, and security code.\n\nPlease be prepared to provide some personal and account information to complete the verification process. Can you please tell me your name, address, and the last 4 digits of your stolen credit card number?"